# 02 - Data Preprocessing for XGBoost

## Purpose

This notebook prepares the processed Olist order dataset for XGBoost modeling.

Based on the data-understanding and EDA stage, this notebook will:
- Load the processed dataset.
- Select the features required for modeling.
- Separate the classification and regression targets.
- Exclude identifier and outcome/leakage columns.
- Handle missing values.
- Transform date columns into usable model features.
- Encode categorical features.
- Prepare the final ML-ready dataset for XGBoost training.

## Feature Decisions

### Selected Features
- customer_state
- primary_seller_state
- primary_category
- item_count
- product_count
- seller_count
- total_price
- total_freight_value
- total_order_value
- avg_item_price
- avg_product_weight_g
- avg_product_length_cm
- avg_product_height_cm
- avg_product_width_cm
- avg_shipping_limit_days
- purchase_year
- purchase_month
- purchase_dayofweek
- order_purchase_timestamp
- order_estimated_delivery_date

### Excluded Columns
- `order_id` — identifier, not a useful model feature
- `customer_id` — identifier, not a useful model feature
- `order_delivered_customer_date` — actual outcome information; causes data leakage
- `delivery_delay_days` — regression target
- `delayed` — classification target

### Targets
- Classification target: `delayed`
- Regression target: `delivery_delay_days`

The final processed data from this notebook will be used by the XGBoost training notebook.

In [17]:
#Import the libraries needed for preprocessing
import pandas as pd
import numpy as np

In [18]:
#Load the processed order dataset
df = pd.read_csv("../../data/processed/processed_orders.csv")

In [19]:
#Check that the dataset loaded correctly
print("Dataset shape:", df.shape)
df.head(2)

Dataset shape: (96476, 25)


,order_id,customer_id,customer_state,primary_seller_state,primary_category,item_count,product_count,seller_count,total_price,total_freight_value,...,avg_product_width_cm,avg_shipping_limit_days,purchase_year,purchase_month,purchase_dayofweek,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date,delivery_delay_days,delayed
0,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,SP,PR,health_beauty,3,1,1,134.97,8.49,...,16.0,4.455,2016,9,3,2016-09-15 12:16:38,2016-10-04,2016-11-09 07:47:38,36.325,1
1,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,SP,PR,watches_gifts,1,1,1,29.90,15.56,...,16.0,18.280,2016,10,0,2016-10-03 09:44:50,2016-10-27,2016-10-26 14:02:13,-0.415,0


In [20]:
#Define the finalized ML features and prediction targets
feature_cols = [
    "customer_state",
    "primary_seller_state",
    "primary_category",
    "item_count",
    "product_count",
    "seller_count",
    "total_price",
    "total_freight_value",
    "total_order_value",
    "avg_item_price",
    "avg_product_weight_g",
    "avg_product_length_cm",
    "avg_product_height_cm",
    "avg_product_width_cm",
    "avg_shipping_limit_days",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

classification_target = "delayed"
regression_target = "delivery_delay_days"

#Verify the number of finalized features
print("Number of features:", len(feature_cols))

Number of features: 20


In [21]:
#Create the feature dataset and separate prediction targets
X = df[feature_cols].copy()
y_class = df[classification_target].copy()
y_reg = df[regression_target].copy()

#Check the dimensions of the features and targets
print("Features:", X.shape)
print("Classification target:", y_class.shape)
print("Regression target:", y_reg.shape)

Features: (96476, 20)
Classification target: (96476,)
Regression target: (96476,)


In [ ]:
# Convert timestamps into numerical date features
for col in timestamp_cols:
    X[col] = pd.to_datetime(X[col])

    X[f"{col}_year"] = X[col].dt.year
    X[f"{col}_month"] = X[col].dt.month
    X[f"{col}_day"] = X[col].dt.day
    X[f"{col}_dayofweek"] = X[col].dt.dayofweek

X = X.drop(columns=timestamp_cols)

print("New feature count:", X.shape[1])             ----cell9

New feature count: 26


In [25]:
#Handle missing values in categorical and numerical features
categorical_cols = X.select_dtypes(include=["object", "category"]).columns
numerical_cols = X.select_dtypes(include=["number"]).columns

X[categorical_cols] = X[categorical_cols].fillna("Unknown")

for col in numerical_cols:
    X[col] = X[col].fillna(X[col].median())

C:\Users\DELL\AppData\Local\Temp\ipykernel_15924\1829318955.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "category"]).columns


In [26]:
# Confirm that all feature values are now complete
print("Total missing values:", X.isnull().sum().sum())

Total missing values: 0


In [ ]:
#Encode categorical features using one-hot for encoding customer_state, primary_seller_state, primary_category
X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=False,
    dtype=int
)

In [29]:
#Check the final feature matrix after categorical encoding
print("Final feature count:", X.shape[1])
print("Dataset shape:", X.shape)

Final feature count: 146
Dataset shape: (96476, 146)


In [30]:
#Prepare the classification and regression targets
y_class = y_class.astype(int)
y_reg = y_reg.astype(float)

print("Classification target shape:", y_class.shape)
print("Regression target shape:", y_reg.shape)

Classification target shape: (96476,)
Regression target shape: (96476,)
